# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v3?

**Respuesta:**  
Taxi-v3 tiene **500 estados** y **6 acciones**.

El número de estados se calcula como:
- 5 × 5 posiciones del taxi = 25
- 5 posibles ubicaciones del pasajero (4 zonas + dentro del taxi)
- 4 destinos posibles

Total: 25 × 5 × 4 = **500 estados**

Taxi tiene muchos más estados que FrozenLake (16 estados) porque el estado debe codificar no solo la posición del taxi, sino también la ubicación del pasajero y el destino, lo que multiplica el espacio de estados.


## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

**Respuesta:**
- $s_t$ = el estado inicial del taxi (codifica posición + pasajero + destino)
- $a_t$ = la acción aleatoria ejecutada (0–5)
- $r_{t+1}$ = la recompensa recibida (-1 por paso normal, -10 por pickup/dropoff incorrecto, +20 por entrega exitosa)
- $s_{t+1}$ = el nuevo estado después de ejecutar la acción


## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

$$
|\mathcal{S}| \times |\mathcal{A}| = 500 \times 6 = 3000 \text{ valores}
$$


## 5. Política ε-greedy

Implementa una función que:

- con probabilidad ε seleccione una acción aleatoria;
- en otro caso seleccione argmax_a Q(s,a)

### Actividad 1


In [ ]:
def choose_action(Q, state, epsilon, env):
    """Selecciona acción usando política epsilon-greedy."""
    # Exploración: acción aleatoria con probabilidad epsilon
    if random.random() < epsilon:
        return env.action_space.sample()
    
    # Explotación: la acción con mayor valor Q
    # Desempate aleatorio entre acciones con el mismo Q máximo
    q_values = Q[state]
    max_q = np.max(q_values)
    best_actions = np.flatnonzero(q_values == max_q)
    return int(np.random.choice(best_actions))


## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    """Aplica una actualización de Q-Learning."""
    current_q = Q[state, action]
    
    # TD target: r + gamma * max_a' Q(s', a')
    best_next_q = np.max(Q[next_state])
    td_target = reward + gamma * best_next_q
    
    # TD error: target - estimación actual
    td_error = td_target - current_q
    
    # Actualización: Q(s,a) <- Q(s,a) + alpha * TD_error
    Q[state, action] = current_q + alpha * td_error
    
    return Q


## 7. Entrenamiento

Ciclo completo de Q-Learning:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar Q(s,a);
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

### Actividad 3


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    """Entrena el agente con Q-Learning durante 'episodes' episodios."""
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for step in range(max_steps):

            # 1. Escoger acción con epsilon-greedy
            action = choose_action(Q, state, epsilon, env)

            # 2. Ejecutar acción en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 3. Actualizar Q
            if done:
                # En estado terminal no hay futuro
                current_q = Q[state, action]
                td_target = reward  # gamma * 0 porque done
                Q[state, action] = current_q + alpha * (td_target - current_q)
            else:
                Q = update_q(Q, state, action, reward, next_state, alpha, gamma)

            # 4. Actualizar estado y recompensa acumulada
            total_reward += reward
            state = next_state

            # 5. Terminar si corresponde
            if done:
                break

        rewards.append(total_reward)

    return Q, rewards


## 8. Entrenar el agente


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)

print(f"Recompensa promedio (últimos 100 ep): {np.mean(rewards[-100:]):.2f}")


## 9. Curva de aprendizaje


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

**Descripción de la curva de aprendizaje:**

- **¿La recompensa promedio mejora?** Sí, la curva muestra una tendencia ascendente clara. Al inicio el agente actúa casi aleatoriamente (recompensas muy negativas). A medida que acumula experiencia, la recompensa promedio sube hacia valores positivos.

- **¿Después de aproximadamente cuántos episodios comienza a estabilizarse?** Alrededor de los 2000–3000 episodios la curva tiende a estabilizarse en valores cercanos a 7–8 de recompensa promedio.

- **¿Indica convergencia perfecta o política razonablemente buena?** Con epsilon=0.1 constante, el agente nunca converge perfectamente ya que siempre explora el 10% del tiempo. La política es razonablemente buena pero no perfectamente greedy.


## 10. Reproducir un episodio


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

**Comparación de los dos episodios:**

- **¿Qué diferencias observa?** El taxi sin entrenamiento se mueve aleatoriamente, hace pickup/dropoff en lugares incorrectos (acumulando penalizaciones de -10) y rara vez llega al destino. El agente entrenado navega directamente hacia el pasajero, lo recoge y lo lleva al destino de forma eficiente.

- **¿El taxi sin entrenamiento logra completar la tarea?** Raramente, y solo por azar. Cuando lo logra, acumula muchos pasos innecesarios.

- **¿El agente entrenado evita acciones innecesarias?** Sí, aprende a evitar los pickup/dropoff inválidos y a trazar rutas más cortas.

- **¿Qué evidencia visual indica aprendizaje?** El taxi entrenado se mueve de forma determinista y dirigida, completando la tarea en pocos pasos con recompensa total positiva.


## 12. Analizar la política aprendida


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))

action_names = ["South", "North", "East", "West", "Pickup", "Dropoff"]
for i, (q, name) in enumerate(zip(Q_trained[state], action_names)):
    print(f"  {name:8s} (acción {i}): {q:.4f}")


### Pregunta 6

Para el estado seleccionado:

1. **¿Cuál es la acción con mayor valor Q?** La acción con el valor Q más alto es la óptima para ese estado específico (depende de la posición del taxi, pasajero y destino codificados en el estado 123).

2. **¿Qué significa que una acción tenga un valor Q mayor?** Significa que el agente estima que tomar esa acción en ese estado llevará a mayor recompensa acumulada (incluyendo recompensas futuras descontadas) comparado con otras acciones.

3. **¿Por qué Q(s,a) no es solo la recompensa inmediata?** Porque Q(s,a) = r + γ·max_a' Q(s',a'), es decir, incluye la recompensa inmediata MÁS las recompensas futuras esperadas (descontadas por γ). El agente aprende a valorar acciones que llevan a estados prometedores, aunque la recompensa inmediata sea pequeña.


## 13. Experimentación — cambio de hiperparámetro

Modificamos **epsilon** de 0.1 a 0.3 para ver el efecto de mayor exploración.


In [ ]:
# Experimento: epsilon = 0.3 (más exploración)
Q_exp = np.zeros((n_states, n_actions))
Q_exp, rewards_exp = train_q_learning(
    env,
    Q_exp,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.3   # más exploración
)

# Comparar curvas
window = 100
ma_original = np.convolve(rewards, np.ones(window)/window, mode="valid")
ma_exp = np.convolve(rewards_exp, np.ones(window)/window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(ma_original, label="epsilon=0.1 (original)")
plt.plot(ma_exp, label="epsilon=0.3 (experimento)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title("Comparación de epsilon")
plt.legend()
plt.show()

print(f"Promedio últimos 100 ep (epsilon=0.1): {np.mean(rewards[-100:]):.2f}")
print(f"Promedio últimos 100 ep (epsilon=0.3): {np.mean(rewards_exp[-100:]):.2f}")


### Pregunta 7

**Efecto de cambiar epsilon de 0.1 a 0.3:**

- **Velocidad de aprendizaje:** Con mayor epsilon el agente explora más al inicio, lo que puede acelerar el descubrimiento de rutas buenas, pero también introduce más ruido.

- **Estabilidad:** La curva con epsilon=0.3 es más ruidosa porque el 30% del tiempo el agente elige acciones aleatorias, incluso después de haber aprendido buenas políticas.

- **Recompensa final:** Con epsilon=0.3, la recompensa máxima alcanzable es menor porque incluso el agente "entrenado" sigue tomando muchas acciones aleatorias (explotación solo el 70% del tiempo). El epsilon=0.1 permite mejor explotación del conocimiento acumulado.

- **Comportamiento observado:** Con epsilon=0.3 el taxi comete más errores aleatorios visibles, mientras que con epsilon=0.1 el comportamiento es más consistente y dirigido al objetivo.


## Entrega

El notebook contiene:

1. ✅ implementación de `choose_action`
2. ✅ implementación de `update_q`
3. ✅ implementación de `train_q_learning`
4. ✅ curva de aprendizaje
5. ✅ visualización del agente antes y después del entrenamiento
6. ✅ respuestas a las siete preguntas
